In [1]:
import torch
import torch.nn as nn
from torch.nn import functional as F

In [2]:
batch_size = 32
block_size = 8

max_iters = 5000
eval_interval = 500

learning_rate = 1e-3

eval_iters = 200

n_embd = 32
head_size = 32

device = "cuda" if torch.cuda.is_available() else "cpu"

torch.manual_seed(1337)

In [3]:
with open("input.txt", "r", encoding="utf-8") as f:
    text = f.read()

print("Length of dataset:", len(text))
print(text[:500])

Length of dataset: 1115394
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor


In [4]:
chars=sorted(list(set(text)))
vocab_size=len(chars)
print(chars)
print(vocab_size)

['\n', ' ', '!', '$', '&', "'", ',', '-', '.', '3', ':', ';', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']
65


In [5]:
stoi = {ch:i for i,ch in enumerate(chars)}
itos = {i:ch for i,ch in enumerate(chars)}

def encode(s):
    return [stoi[c] for c in s]

def decode(l):
    return ''.join([itos[i] for i in l])

In [6]:
data = torch.tensor(encode(text), dtype=torch.long)

print(data.shape)
print(data[:100])

torch.Size([1115394])
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59])


In [7]:
n=int(0.9*len(data))

train_data=data[:n]
val_data=data[n:]

print(len(train_data))
print(len(val_data))

1003854
111540


In [8]:
def get_batch(split):
    data=train_data if split=="train" else val_data
    ix=torch.randint(len(data)-block_size,(batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)

    return x, y

In [9]:
@torch.no_grad()
def estimate_loss():

    out = {}

    model.eval()

    for split in ["train", "val"]:

        losses = torch.zeros(eval_iters)

        for k in range(eval_iters):

            X, Y = get_batch(split)

            logits, loss = model(X, Y)

            losses[k] = loss.item()

        out[split] = losses.mean()

    model.train()

    return out

In [10]:
class Head(nn.Module):

    def __init__(self, n_embd, head_size):

        super().__init__()

        self.key = nn.Linear(
            n_embd,
            head_size,
            bias=False
        )

        self.query = nn.Linear(
            n_embd,
            head_size,
            bias=False
        )

        self.value = nn.Linear(
            n_embd,
            head_size,
            bias=False
        )

        self.register_buffer(
            "tril",
            torch.tril(
                torch.ones(
                    block_size,
                    block_size
                )
            )
        )

    def forward(self, x):

        B,T,C = x.shape

        k = self.key(x)

        q = self.query(x)

        v = self.value(x)

        wei = q @ k.transpose(-2,-1)

        wei = wei * (k.size(-1) ** -0.5)

        wei = wei.masked_fill(
            self.tril[:T,:T] == 0,
            float("-inf")
        )

        wei = F.softmax(
            wei,
            dim=-1
        )

        out = wei @ v

        return out

In [11]:
class AttentionLanguageModel(nn.Module):

    def __init__(self):

        super().__init__()

        self.token_embedding_table = nn.Embedding(
            vocab_size,
            n_embd
        )

        self.position_embedding_table = nn.Embedding(
            block_size,
            n_embd
        )

        self.sa_head = Head(
            n_embd,
            head_size
        )

        self.lm_head = nn.Linear(
            head_size,
            vocab_size
        )

    def forward(
        self,
        idx,
        targets=None
    ):

        B,T = idx.shape

        tok_emb = self.token_embedding_table(idx)

        pos_emb = self.position_embedding_table(
            torch.arange(
                T,
                device=device
            )
        )

        x = tok_emb + pos_emb

        x = self.sa_head(x)

        logits = self.lm_head(x)

        if targets is None:

            loss = None

        else:

            B,T,C = logits.shape

            logits = logits.view(
                B*T,
                C
            )

            targets = targets.view(B*T)

            loss = F.cross_entropy(
                logits,
                targets
            )

        return logits, loss

    def generate(
        self,
        idx,
        max_new_tokens
    ):

        for _ in range(max_new_tokens):

            idx_cond = idx[:, -block_size:]

            logits, loss = self(idx_cond)

            logits = logits[:, -1, :]

            probs = F.softmax(
                logits,
                dim=-1
            )

            idx_next = torch.multinomial(
                probs,
                num_samples=1
            )

            idx = torch.cat(
                (idx, idx_next),
                dim=1
            )

        return idx

In [12]:
model = AttentionLanguageModel()

model = model.to(device)

print(
    sum(p.numel() for p in model.parameters()) / 1e3,
    "K parameters"
)

7.553 K parameters


In [13]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=learning_rate
)

In [14]:
for iter in range(max_iters):

    if iter % eval_interval == 0:

        losses = estimate_loss()

        print(
            f"step {iter}: "
            f"train loss {losses['train']:.4f}, "
            f"val loss {losses['val']:.4f}"
        )

    xb, yb = get_batch("train")

    logits, loss = model(xb, yb)

    optimizer.zero_grad(set_to_none=True)

    loss.backward()

    optimizer.step()

step 0: train loss 4.2000, val loss 4.2047
step 500: train loss 2.6911, val loss 2.7087
step 1000: train loss 2.5196, val loss 2.5303
step 1500: train loss 2.4775, val loss 2.4829
step 2000: train loss 2.4408, val loss 2.4523
step 2500: train loss 2.4272, val loss 2.4435
step 3000: train loss 2.4130, val loss 2.4327
step 3500: train loss 2.3956, val loss 2.4212
step 4000: train loss 2.4041, val loss 2.3992
step 4500: train loss 2.3980, val loss 2.4084


In [15]:
context = torch.zeros(
    (1,1),
    dtype=torch.long,
    device=device
)

generated_text = decode(
    model.generate(
        context,
        max_new_tokens=500
    )[0].tolist()
)

print(generated_text)


K:
NGey

Letnrad wineam:
Kicou hitipteavimancraby whet muthe hus darge.

Wind!
IRD: Ind, tind spoof om and f.
Sy stllalevere here me honouen fot in,
So and, vist orby?
Thar hous mat deest she rd?

Wowin wof t, ath th ay miligiryouchth-orto mou tenges, ald pors banebe y prothetack aklel I veriplansnidierd avit for,
KI thit ndist allll perd the:
Acu Empoouthant, I to
Ten mar.

S:
Bugh the I hy nd meis moh h!


AThamen es ty I has.

MI ithe thensterat blo gaar,
A d muts ed ronur wiend tl-ou,
Therim


In [16]:
context = torch.zeros((1,1), dtype=torch.long, device=device)

sample = decode(
    model.generate(
        context,
        max_new_tokens=200
    )[0].tolist()
)

print(sample)


Y: waenche edrth,
NG ive weiteve.
Thave ndetrid wifrud has is.
 JUS:
Th
Misho om sod winom bulall kangesoue be yorime hacigu tha ph med mpeat oead out acarmitso sas
thentesein,
Y:
Tel sst forsheriy an
